# LLM講座2 最終課題：Colab 評価ノートブック（開発用セット・vLLM）

会社 PC の `run_experiments.py` / `yesno_all_runs.py` / `analyze_errors*.py` / `check_train_data.py` を、**Drive のパスで動く Colab 版**にまとめたもの。

**流れ**
1. （別ノートブック）R01 のコピーで、開発用セットの検索結果を `LLM講座最終コンペ_eval/dev_contexts.jsonl` に1回だけ保存する（手順書の「手順B」）
2. このノートブック：検索だけの評価（R@k）→ 生成（vLLM）→ 採点 → CSV 追記 → 集計・誤り分析・2本の比較

**守ること**
- 開発用セット（NIILC・JEMHopQA 由来）だけを扱う。本番の `questions.json` / `submission.json` には触れない（下のセルでパスを検査して止める）
- 上から順に実行する。「実行」セルは RUN_NAMES を確認してから

In [42]:
# # [step0.1 setup-install]  ←  このセルの名前（以降のセルも1行目に名前を記載）
# VLLM_WHEEL = "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
# !printf "torch==2.11.0\ntorchvision==0.26.0\n" > /tmp/constraints.txt
# !pip install -q "$VLLM_WHEEL" "sudachipy==0.6.11" "sudachidict-core==20260428" -c /tmp/constraints.txt > /tmp/pip_install.log 2>&1
# print("exit code:", _exit_code)   # 直前の pip install の終了コード（0 なら成功）
# !pip uninstall -y -q torchcodec
# !tail -3 /tmp/pip_install.log

In [43]:
# [step0.1 setup-install]  ← 一番上（E1 より前）に置く。入っていれば飛ばし、入れるときは各段階の秒数を表示する
import importlib.metadata as _md, subprocess, time

def _ver(p):
    try:
        return _md.version(p)
    except _md.PackageNotFoundError:
        return None

if (_ver("vllm") or "").startswith("0.26.0") and (_ver("torch") or "").startswith("2.11.0") \
        and _ver("sudachipy") == "0.6.11" and _ver("sudachidict-core") == "20260428":
    print("インストール済みなので飛ばす: vllm %s / torch %s" % (_ver("vllm"), _ver("torch")))
else:
    VLLM_WHEEL = "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
    def _run(label, cmd):
        t = time.time()
        rc = subprocess.run(cmd, shell=True).returncode
        print("[%s] %.0f 秒（終了コード %d）" % (label, time.time() - t, rc))
        return rc
    open("/tmp/constraints.txt", "w").write("torch==2.11.0\ntorchvision==0.26.0\n")
    rc = _run("pip install", 'pip install -q "%s" "sudachipy==0.6.11" "sudachidict-core==20260428" -c /tmp/constraints.txt > /tmp/pip_install.log 2>&1' % VLLM_WHEEL)
    _run("uninstall torchcodec", "pip uninstall -y -q torchcodec")
    print(subprocess.run("tail -3 /tmp/pip_install.log", shell=True, capture_output=True, text=True).stdout)
    if rc != 0:
        raise SystemExit("pip install に失敗した（終了コード %d）。/tmp/pip_install.log の末尾を貼ってください" % rc)
    print("インストールした。このノートブックは vLLM を別プロセスで動かすので、ランタイムの再起動は不要（E7 で確認）")

インストール済みなので飛ばす: vllm 0.26.0+cu129 / torch 2.11.0+cu128


In [44]:
# [E1 設定] Drive のパス・フォルダ作成（最初に必ず実行）
import os, sys, json, time, datetime, hashlib, shutil, subprocess, re, unicodedata, collections, csv, glob

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
assert os.path.isdir("/content/drive/MyDrive"), "Drive が見えない。ランタイムを接続し直して、このセルをもう一度実行する"

MY        = "/content/drive/MyDrive"
EVAL_ROOT = MY + "/LLM講座最終コンペ_eval"          # 評価専用フォルダ（新規。既存のフォルダは触らない）
LORA_ROOT = MY + "/LLM講座最終コンペ_lora"          # 画面にある学習用フォルダ（読むだけ）
DEV_ANSWERS  = EVAL_ROOT + "/dev_data/dev_answers.json"   # 開発用セットの正解（会社 PC の dev_data から上げる）
DEV_CONTEXTS = EVAL_ROOT + "/dev_contexts.jsonl"          # 手順Bで作る：開発用セットの検索結果（資料10件）
RUNS      = EVAL_ROOT + "/runs"                      # 各実験の出力
RUNNER    = EVAL_ROOT + "/_runner"
RESULTS_CSV = RUNNER + "/experiments_results.csv"    # 実験ごとの結果（追記）
TOOLS     = EVAL_ROOT + "/tools"                     # scoring.py を置く場所・worker スクリプト

for d in (EVAL_ROOT + "/dev_data", EVAL_ROOT + "/dev_work", RUNS, RUNNER, TOOLS):
    os.makedirs(d, exist_ok=True)

def guard_path(p):
    # 本番のフォルダ（…コンペ_work）を読む・書くのを止める（質問ファイル・提出ファイルに触れない）
    if "LLM講座最終コンペ_work" in str(p):
        raise SystemExit("本番用フォルダのパスは使わない: " + str(p))
    return p

for _p in (DEV_ANSWERS, DEV_CONTEXTS, RUNS, RESULTS_CSV, TOOLS):
    guard_path(_p)

print("EVAL_ROOT :", EVAL_ROOT)
print("dev_answers.json   :", "あり" if os.path.exists(DEV_ANSWERS) else "【なし】dev_data にアップロードする")
print("dev_contexts.jsonl :", "あり" if os.path.exists(DEV_CONTEXTS) else "【なし】手順Bで作る")
print("scoring.py         :", "あり" if os.path.exists(TOOLS + "/scoring.py") else "なし（簡易版で採点する。会社 PC の scoring.py を tools/ に上げると同じ採点になる）")

EVAL_ROOT : /content/drive/MyDrive/LLM講座最終コンペ_eval
dev_answers.json   : あり
dev_contexts.jsonl : あり
scoring.py         : あり


In [45]:
# [E2 環境確認] GPU と vLLM
import subprocess
import sys
import importlib.metadata as md_
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
GPU = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "なし"
print("python :", sys.version.split()[0])
print("GPU    :", GPU)
for p in ("vllm", "torch", "transformers"):
    try:
        print("%-13s %s" % (p, md_.version(p)))
    except md_.PackageNotFoundError:
        print("%-13s なし" % p)
if "なし" == GPU:
    print("\n【注意】GPU がない。検索だけの評価（E4）は動くが、生成（E7）は GPU のランタイムが必要。L4 を推奨（T4 でも fp16 で動く）")
try:
    md_.version("vllm")
except md_.PackageNotFoundError:
    print("\n【vLLM がない】R01 の [step0.1] セルをこのセルの前に貼って実行し、ランタイムを再起動してから E1 に戻る。")
    print("（照合用：提出環境は python 3.12.13 / torch 2.11.0+cu128 / transformers 5.13.1 / vllm 0.26.0+cu129。生成を動かすまでは、このまま E4 まで進めてよい）")

python : 3.12.13
GPU    : Tesla T4, 15360 MiB
vllm          0.26.0+cu129
torch         2.11.0+cu128
transformers  5.13.1


In [46]:
# [E3 採点] scoring.py があればそれを使う。なければ簡易版（EM・部分一致）
import sys
import os
import json
import re
import unicodedata
import importlib

# TOOLS が未定義の場合はデフォルト値をフォールバックとして定義（E1 セルが未実行の場合対策）
if "TOOLS" not in globals():
    TOOLS = "/content/drive/MyDrive/LLM講座最終コンペ_eval/tools"

sys.path.insert(0, TOOLS)
SCORING_SRC = None
try:
    import scoring
    importlib.reload(scoring)
    yesno, judge, norm = scoring.yesno, scoring.judge, scoring.norm
    SCORING_SRC = "tools/scoring.py（会社 PC と同じ採点）"
except Exception as e:
    def norm(s):
        s = unicodedata.normalize("NFKC", str(s or "")).lower()
        return re.sub(r"[\W_]+", "", s)
    def yesno(s):
        n = norm(s)
        if n in ("はい", "yes"): return "yes"
        if n in ("いいえ", "no"): return "no"
        return None
    def judge(pred, golds):
        p = norm(pred)
        gn = [norm(a) for a in golds if norm(a)]
        if not gn or not p:
            return "不正解"
        yn_gold = all(g in ("yes", "no") for g in gn)
        if yn_gold:
            return "完全一致" if yesno(pred) in gn else "不正解"
        if any(p == g for g in gn):
            return "完全一致"
        if any(g in p or p in g for g in gn):
            return "部分一致"
        return "不正解"
    SCORING_SRC = "簡易版（scoring.py が tools/ にない: %s）" % type(e).__name__
print("採点:", SCORING_SRC)
if not SCORING_SRC.startswith("tools"):
    print("※ 簡易版の数字は、会社 PC の数字と 1〜数問ずれることがある。会社 PC の code\\scoring.py を tools/ に上げて、このセルをやり直すと揃う")

def load_gold(path):
    g = json.load(open(guard_path(path), encoding="utf-8"))
    out = {}
    if isinstance(g, dict):
        for k, v in g.items():
            if isinstance(v, dict):
                a = v.get("answers", v.get("answer")); d = dict(v)
            else:
                a, d = v, {}
            d["answers"] = [a] if isinstance(a, str) else list(a or [])
            out[k] = d
    elif isinstance(g, list):
        for r in g:
            a = r.get("answers", r.get("answer")); d = dict(r)
            d["answers"] = [a] if isinstance(a, str) else list(a or [])
            out[r["q_id"]] = d
    return out

def is_yn(golds):
    return bool(golds) and set(str(a).upper() for a in golds) <= {"YES", "NO"}

def load_jsonl(path):
    with open(guard_path(path), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

採点: tools/scoring.py（会社 PC と同じ採点）


In [47]:
# # [E4診断] q_id の食い違いを調べる（ID と件数だけ表示。質問文・正解は表示しない）
# g_ids = list(GOLD.keys())
# c_ids = [r["q_id"] for r in DEV_ROWS]
# def _types(xs): return dict(collections.Counter(type(x).__name__ for x in xs))
# print("正解側   : %d 件 / 型 %s / 先頭5: %s" % (len(g_ids), _types(g_ids), g_ids[:5]))
# print("検索結果 : %d 件 / 型 %s / 先頭5: %s" % (len(c_ids), _types(c_ids), c_ids[:5]))
# print("文字列にして一致        :", len({str(x) for x in g_ids} & {str(x) for x in c_ids}))
# _dg = lambda x: re.sub(r"\D", "", str(x))
# print("数字だけにして一致      :", len({_dg(x) for x in g_ids} & {_dg(x) for x in c_ids}))
# gq = {norm(v.get("question")) for v in GOLD.values() if v.get("question")}
# cq = {norm(r["question"]) for r in DEV_ROWS}
# print("正解側に質問文あり      : %d 件" % len(gq))
# print("質問文が一致            : %d 件（検索結果 %d 問中）" % (len(gq & cq), len(cq)))
# qp = EVAL_ROOT + "/dev_work/questions.json"
# if os.path.exists(qp):
#     _q = json.load(open(qp, encoding="utf-8"))
#     _q = list(_q.values()) if isinstance(_q, dict) else _q
#     _ids = [x.get("q_id", x.get("id")) for x in _q]
#     print("dev_work/questions.json : %d 件 / 先頭5: %s" % (len(_ids), _ids[:5]))
#     print("  検索結果の q_id と一致 :", len({str(x) for x in _ids} & {str(x) for x in c_ids}))
#     print("  正解側の q_id と一致   :", len({str(x) for x in _ids} & {str(x) for x in g_ids}))
# else:
#     print("dev_work/questions.json : なし")

In [48]:
# [E4 検索だけの評価] dev_contexts.jsonl の R@k（GPU 不要・数秒）
GOLD = load_gold(DEV_ANSWERS)
DEV_ROWS = load_jsonl(DEV_CONTEXTS)
print("正解:", len(GOLD), "問 / 検索結果:", len(DEV_ROWS), "問 / 1問あたりの資料:", collections.Counter(len(r["contexts"]) for r in DEV_ROWS))
missing = [r["q_id"] for r in DEV_ROWS if r["q_id"] not in GOLD]
print("正解がない q_id:", len(missing), "（0 のはず）")
assert not missing, "dev_answers.json と dev_contexts.jsonl の q_id が合わない"
assert all(len(r["contexts"]) >= 10 for r in DEV_ROWS), "資料が10件未満の問題がある。手順Bの CONTEXT_TOP_K = 10 を確認"

def has_gold(ctxs, golds):
    gn = [norm(a) for a in golds if norm(a)]
    return any(g in norm(c["title"] + c["text"]) for g in gn for c in ctxs)

def retrieval_metrics(rows, ks=(1, 3, 5, 10)):
    # 正解の文字列を含む資料が上位 k 件にある割合。はい/いいえ問題は除く（会社 PC の R@k と同じ定義）
    tgt = [r for r in rows if not is_yn(GOLD[r["q_id"]]["answers"])]
    res = {"all_n": len(tgt)}
    for k in ks:
        res["all_R@%d" % k] = round(sum(has_gold(r["contexts"][:k], GOLD[r["q_id"]]["answers"]) for r in tgt) / max(len(tgt), 1), 4)
    return res

RET = retrieval_metrics(DEV_ROWS)
print(RET)
print("参考（会社 PC・R00_B1_check・候補100件・資料10件・n=275）: R@1 0.4836 / R@3 0.6218 / R@5 0.6873 / R@10 0.7273")
print("→ これと数問（±0.01 前後）以内なら、Colab の検索が会社 PC と同じ動きをしている")

正解: 320 問 / 検索結果: 320 問 / 1問あたりの資料: Counter({10: 320})
正解がない q_id: 0 （0 のはず）
{'all_n': 275, 'all_R@1': 0.4873, 'all_R@3': 0.6255, 'all_R@5': 0.6909, 'all_R@10': 0.7345}
参考（会社 PC・R00_B1_check・候補100件・資料10件・n=275）: R@1 0.4836 / R@3 0.6218 / R@5 0.6873 / R@10 0.7273
→ これと数問（±0.01 前後）以内なら、Colab の検索が会社 PC と同じ動きをしている


In [49]:
# [E5 実験の一覧] experiments_plan.py の移植（検索は dev_contexts.jsonl で固定。生成モデル・資料件数・プロンプトを変える実験だけ）
MODELS = {
    "base":  dict(kind="hf", repo="Qwen/Qwen3.5-0.8B",                         revision="2fc06364715b967f1860aea9cf38778875588b17"),
    "final": dict(kind="hf", repo="YAMAHARZ250/qwen3.5-0.8b-rag-lora-final",   revision="9a4d0723a493aa3c2b9570b065744ea16f4a9237"),
    "v2s1":  dict(kind="hf", repo="YAMAHARZ250/qwen3.5-0.8b-rag-lora-v2s1",    revision="e01f534af4b9e04d93ae001e06faa7d306fd7fbb"),
    "v2s2":  dict(kind="hf", repo="YAMAHARZ250/qwen3.5-0.8b-rag-lora-v2s2",    revision="52588dd277ec1531eb951d45573564d3dceb5d49"),
    # v2t0 は revision が未記録なので Drive の変換済みモデルを使う（画面の train_v2/v2_t0/vllm）
    "v2t0":  dict(kind="drive", path=LORA_ROOT + "/train_v2/v2_t0/vllm"),
    # 今後の例（重み平均など）: "soup1": dict(kind="drive", path=LORA_ROOT + "/train_v2/soup1/vllm"),
}
BEST_GEN = {"temperature": 0.0, "presence_penalty": 0.0, "max_new_tokens": 64}   # 提出と同じ生成設定
P2 = "P2_short_qfirst"

def E(name, model, ctx_k=5, limit=None, gen=None, prompt=P2, note=""):
    return {"name": name, "model": model, "ctx_k": ctx_k, "limit": limit,
            "gen": dict(BEST_GEN, **(gen or {})), "prompt": prompt, "note": note}

EXPERIMENTS = [
    E("C_S03_base_ctx5_320",    "base",  5,  note="対照：元のモデル・資料5件（LoRA の効果＝L10 との差）"),
    E("C_L10_final_ctx5_320",   "final", 5,  note="基準：提出版 lora-final・資料5件（本番 0.4186 と同じ構成）。会社 PC では EM 0.3969（127問）"),
    E("C_L11_final_ctx7_320",   "final", 7,  note="L10 の資料を7件に（学習は5件のまま）"),
    E("C_L12_final_ctx10_320",  "final", 10, note="L10 の資料を10件に（学習は5件のまま）"),
    E("C_V21_s1_ctx5_320",      "v2s1",  5,  note="LoRA v2・種1・はい/いいえは1件目"),
    E("C_V22_s2_ctx5_320",      "v2s2",  5,  note="LoRA v2・種2（本番 0.41196 と同じ構成）。会社 PC では EM 0.4437（142問）"),
    E("C_V23_t0_ctx5_320",      "v2t0",  5,  note="LoRA v2・種0・はい/いいえは記事名の資料"),
]
# 移植しなかったもの：
#   L13（リランク候補200件）・V03r・E2a/E2b … 検索そのものを変える実験。dev_contexts.jsonl を作り直す（手順B）必要がある
#   L14 … 会社 PC で学習したモデルを使う実験（Colab 評価には使わない）
print("実験一覧:", [e["name"] for e in EXPERIMENTS])

実験一覧: ['C_S03_base_ctx5_320', 'C_L10_final_ctx5_320', 'C_L11_final_ctx7_320', 'C_L12_final_ctx10_320', 'C_V21_s1_ctx5_320', 'C_V22_s2_ctx5_320', 'C_V23_t0_ctx5_320']


In [50]:
EXPERIMENTS.append(E("C_V22_s2_ctx10_320", "v2s2", 10, note="V22を資料10件で（L12の効果がV22でも出るか）"))
RUN_NAMES = ["C_V22_s2_ctx10_320"]
FORCE = False

In [51]:
# [E6 生成の部品] プロンプト P2・vLLM の worker・結果の記録

def answer_prompt(question, contexts):   # P2（提出用ノートブックの [step7.1] と同じ。md5 で照合する）
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "質問: " + question + "\n\n"
        "次の資料を根拠に、上の質問に短く答えてください。\n"
        "答え方のきまり:\n"
        "- answer には、質問の答えになる語句だけを書く（人名・地名・作品名・数値・日付など）。\n"
        "- 文にしない。「〜です」「〜である」を付けない。説明・補足・括弧書き・読み仮名・英語表記を付けない。\n"
        "- 「どちら」「どっち」と聞かれたら、選んだほうの名前だけを書く。\n"
        "- 「はい」か「いいえ」で答える質問には、「はい」か「いいえ」だけを書く。\n"
        "- 資料のどこにも答えが書かれていない場合だけ、answer を「資料からは特定できません」、chunk_id を null にする。\n"
        "chunk_id には、答えが書かれている資料の chunk_id(整数)を1つだけ書く。\n"
        '形式: {"answer":"答えの語句","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問（再掲）: " + question
    )
PROMPTS = {"P2_short_qfirst": answer_prompt}
TRAIN_MD5 = "1e990717ef9ebb51535b233deec1503e"
_now = hashlib.md5(answer_prompt("Q", [{"chunk_id": 1, "title": "T", "text": "X"}]).encode("utf-8")).hexdigest()
print("P2 の md5:", _now, "/ 学習時:", TRAIN_MD5, "→", "一致" if _now == TRAIN_MD5 else "【不一致】")
assert _now == TRAIN_MD5, "プロンプトが学習時と違う。このセルを直さずに先へ進まない"

WORKER = r'''
import json, sys, time
cfg = json.load(open(sys.argv[1], encoding="utf-8"))
from vllm import LLM, SamplingParams
rows = [json.loads(l) for l in open(cfg["prompts"], encoding="utf-8") if l.strip()]
schema = {"type": "object",
          "properties": {"answer": {"type": "string"}, "chunk_id": {"type": ["integer", "null"]}},
          "required": ["answer", "chunk_id"]}
g = cfg["gen"]
kw = dict(max_tokens=g["max_new_tokens"], temperature=g["temperature"], top_p=1.0, top_k=20, min_p=0.0,
          presence_penalty=g["presence_penalty"], repetition_penalty=1.0)
sp, errs = None, []
try:
    from vllm.sampling_params import StructuredOutputsParams
    sp = SamplingParams(structured_outputs=StructuredOutputsParams(json=schema), **kw)
except Exception as e:
    errs.append(repr(e))
    try:
        from vllm.sampling_params import GuidedDecodingParams
        sp = SamplingParams(guided_decoding=GuidedDecodingParams(json=schema), **kw)
    except Exception as e2:
        errs.append(repr(e2))
if sp is None:
    print("[gen] 構造化出力（JSON スキーマ）が使えない:", errs, flush=True)
    sys.exit(3)
llm_kw = dict(model=cfg["model"], dtype="float16", max_model_len=8192, gpu_memory_utilization=cfg["gpu_util"],
              enforce_eager=True, seed=42, limit_mm_per_prompt={"image": 0, "video": 0})
if cfg.get("revision"):
    llm_kw["revision"] = cfg["revision"]
    llm_kw["tokenizer_revision"] = cfg["revision"]
t_load = time.time()
llm = LLM(**llm_kw)
print("[gen] モデル読み込み %.1f 分" % ((time.time() - t_load) / 60), flush=True)
t0, out = time.time(), []
B = 64
for i in range(0, len(rows), B):
    batch = rows[i:i + B]
    msgs = [[{"role": "user", "content": r["prompt"]}] for r in batch]
    res = llm.chat(msgs, sp, chat_template_kwargs={"enable_thinking": False}, use_tqdm=False)
    for r, o in zip(batch, res):
        out.append({"q_id": r["q_id"], "text": o.outputs[0].text, "finish": o.outputs[0].finish_reason})
    print("[gen] %d/%d（%.1f 分）" % (len(out), len(rows), (time.time() - t0) / 60), flush=True)
with open(cfg["out"], "w", encoding="utf-8") as f:
    for r in out:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("[gen] done %.2f" % ((time.time() - t0) / 60), flush=True)
'''
# open(TOOLS + "/gen_worker.py", "w", encoding="utf-8").write(WORKER)
WORKER_MAIN = "def main():\n" + "\n".join(("    " + l) if l.strip() else l for l in WORKER.splitlines()) + "\n\n\nif __name__ == \"__main__\":\n    main()\n"
open(TOOLS + "/gen_worker.py", "w", encoding="utf-8").write(WORKER_MAIN)


CSV_FIELDS = ["name", "status", "limit", "ctx_k", "gen_n", "gen_EM", "gen_部分一致", "chunk_None", "特定不可",
              "all_n", "all_R@1", "all_R@3", "all_R@5", "all_R@10", "model", "revision", "prompt",
              "minutes", "gen_minutes", "gpu", "scoring", "run_dir", "time", "note"]

def done_names():
    if not os.path.exists(RESULTS_CSV):
        return set()
    with open(RESULTS_CSV, encoding="utf-8-sig") as f:
        return {r["name"] for r in csv.DictReader(f) if r.get("status") == "ok"}

def append_result(row):
    new = not os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow(row)

def resolve_model(key):
    m = MODELS[key]
    if m["kind"] == "hf":
        return m["repo"], m["revision"]
    src = m["path"]
    assert os.path.isfile(src + "/model.safetensors") and os.path.isfile(src + "/config.json"), "モデルが見えない: " + src
    dst = "/content/models/" + key                      # Drive から直接読むと遅いので、ローカルディスクにコピーする
    if not os.path.isfile(dst + "/model.safetensors"):
        os.makedirs("/content/models", exist_ok=True)
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
    return dst, None

def parse_output(text):
    try:
        o = json.loads(text)
        ans = o.get("answer")
        cid = o.get("chunk_id")
        return ("" if ans is None else str(ans)), (cid if isinstance(cid, int) else None)
    except Exception:
        return text.strip(), None

def score_submission(sub):
    rows = [r for r in sub if r["q_id"] in GOLD]
    em = part = 0
    for r in rows:
        j = judge(r.get("answer"), GOLD[r["q_id"]]["answers"])
        em += j == "完全一致"
        part += j in ("完全一致", "部分一致")
    n = max(len(rows), 1)
    return {"gen_n": len(rows), "gen_EM": round(em / n, 4), "gen_部分一致": round(part / n, 4),
            "chunk_None": sum(1 for r in rows if r.get("chunk") is None),
            "特定不可": sum(1 for r in rows if str(r.get("answer") or "").startswith("資料からは特定"))}

def run_experiment(e):
    t_start = time.time()
    name = e["name"]
    rows = DEV_ROWS[:e["limit"]] if e["limit"] else DEV_ROWS
    k = e["ctx_k"]
    assert k <= min(len(r["contexts"]) for r in rows), "dev_contexts.jsonl の資料が足りない"
    model, rev = resolve_model(e["model"])
    run_dir = "%s/%s_%s" % (RUNS, name, datetime.datetime.now().strftime("%Y%m%d_%H%M"))
    os.makedirs(run_dir, exist_ok=True)
    pf = PROMPTS[e["prompt"]]
    with open(run_dir + "/prompts.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({"q_id": r["q_id"], "prompt": pf(r["question"], r["contexts"][:k])}, ensure_ascii=False) + "\n")
    cfg = {"model": model, "revision": rev, "gen": e["gen"], "gpu_util": 0.5,
           "prompts": run_dir + "/prompts.jsonl", "out": run_dir + "/raw.jsonl"}
    cfg_path = run_dir + "/generation_config.json"
    json.dump(dict(cfg, name=name, ctx_k=k, prompt=e["prompt"], model_key=e["model"], limit=e["limit"]),
              open(cfg_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    base = {"name": name, "limit": e["limit"] or len(DEV_ROWS), "ctx_k": k, "model": model, "revision": rev or "",
            "prompt": e["prompt"], "gpu": GPU, "scoring": SCORING_SRC, "run_dir": run_dir, "note": e["note"],
            "time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}
    print("=== %s（モデル %s・資料 %d 件・%d 問）===" % (name, e["model"], k, len(rows)))
    log = []
    proc = subprocess.Popen([sys.executable, TOOLS + "/gen_worker.py", cfg_path], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.append(line)
        if line.startswith("[gen]"):
            print(line.rstrip())
    rc = proc.wait()
    open(run_dir + "/worker.log", "w", encoding="utf-8").write("".join(log))
    if rc != 0 or not os.path.exists(cfg["out"]):
        print("【失敗】終了コード", rc, "/ ログの末尾（原因の行はこの中か、worker.log の上のほうにある）:")
        print("".join(log[-40:]))
        append_result(dict(base, status="error rc=%s" % rc, minutes=round((time.time() - t_start) / 60, 1)))
        return None
    raw = {r["q_id"]: r for r in load_jsonl(cfg["out"])}
    sub = []
    for r in rows:
        ans, cid = parse_output(raw[r["q_id"]]["text"])
        sub.append({"q_id": r["q_id"], "answer": ans, "chunk": cid})
    json.dump(sub, open(run_dir + "/submission.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    res = dict(base, status="ok", minutes=round((time.time() - t_start) / 60, 1),
               **score_submission(sub), **retrieval_metrics(rows))
    gm = [l for l in log if l.startswith("[gen] done")]
    res["gen_minutes"] = gm[-1].split()[-1] if gm else ""
    append_result(res)
    print("→ EM %.4f（%d 問）/ 部分一致 %.4f / chunk=None %d / 特定不可 %d / %.1f 分" %
          (res["gen_EM"], round(res["gen_EM"] * res["gen_n"]), res["gen_部分一致"], res["chunk_None"], res["特定不可"], res["minutes"]))
    return res
print("準備OK")

P2 の md5: 1e990717ef9ebb51535b233deec1503e / 学習時: 1e990717ef9ebb51535b233deec1503e → 一致
準備OK


In [68]:
# [E14 並び順の診断] 資料の順番だけを変えて、V22 をもう一度動かす（診断用。提出には使わない）
#   rev5  : 上位5件を逆順にする（正解の資料が1番目に来にくくなる）
#   gold5 : 正解の文字列を含む資料を1番目に移す（開発用セットの正解を使った「理想の並べ替え」。上限を見るだけ）
ORIG_ROWS = DEV_ROWS

def _has(c, golds):
    gn = [norm(a) for a in golds if norm(a)]
    return any(g in norm(c["title"] + c["text"]) for g in gn)

def reorder(row, mode):
    top = list(row["contexts"][:5])
    if mode == "rev5":
        top = top[::-1]
    elif mode == "gold5" and not is_yn(GOLD[row["q_id"]]["answers"]):
        hit = [i for i, c in enumerate(top) if _has(c, GOLD[row["q_id"]]["answers"])]
        if hit:
            top.insert(0, top.pop(hit[0]))
    return dict(row, contexts=top)

for mode in ("rev5", "gold5"):
    name = "D_V22_s2_%s_320" % mode
    if name in done_names():
        print("飛ばす（実行済み）:", name); continue
    DEV_ROWS = [reorder(r, mode) for r in ORIG_ROWS]
    try:
        run_experiment(E(name, "v2s2", 5, note="診断：資料の並び順=%s（all_R@k の列は並べ替え後の値なので見ない）" % mode))
    finally:
        DEV_ROWS = ORIG_ROWS
print("DEV_ROWS を元に戻した:", DEV_ROWS is ORIG_ROWS)

=== D_V22_s2_rev5_320（モデル v2s2・資料 5 件・320 問）===
[gen] モデル読み込み 0.8 分
[gen] 64/320（0.6 分）
[gen] 128/320（1.2 分）
[gen] 192/320（1.8 分）
[gen] 256/320（2.4 分）
[gen] 320/320（3.0 分）
[gen] done 2.97
→ EM 0.4094（131 問）/ 部分一致 0.4875 / chunk=None 1 / 特定不可 0 / 4.4 分
=== D_V22_s2_gold5_320（モデル v2s2・資料 5 件・320 問）===
[gen] モデル読み込み 0.8 分
[gen] 64/320（0.7 分）
[gen] 128/320（1.3 分）
[gen] 192/320（1.9 分）
[gen] 256/320（2.5 分）
[gen] 320/320（3.0 分）
[gen] done 3.04
→ EM 0.4562（146 問）/ 部分一致 0.5500 / chunk=None 0 / 特定不可 0 / 4.3 分
DEV_ROWS を元に戻した: True


In [52]:
# [E7 実行] 実験を動かす（実行済み＝CSV の status=ok の実験は飛ばす）
# RUN_NAMES = ["C_L10_final_ctx5_320", "C_V22_s2_ctx5_320"]     # ← 動かす実験名（E5 の一覧から選ぶ）。まずこの2本で基準を測り直す

# RUN_NAMES = ["C_V21_s1_ctx5_320", "C_V23_t0_ctx5_320"]
# RUN_NAMES = ["C_L10_final_ctx5_320", "C_L11_final_ctx7_320", "C_L12_final_ctx10_320"]
BASE_RUN, NEW_RUN = "C_L10_final_ctx5_320", "C_L12_final_ctx10_320"

# FORCE = True
RUN_NAMES = ["C_V22_s2_ctx10_320"]
FORCE = False
                                              # True: 実行済みでも、もう一度動かす

unknown = [n for n in RUN_NAMES if n not in {e["name"] for e in EXPERIMENTS}]
assert not unknown, "E5 にない実験名: %s" % unknown
for e in EXPERIMENTS:
    if e["name"] in RUN_NAMES:
        if e["name"] in done_names() and not FORCE:
            print("飛ばす（実行済み）:", e["name"])
            continue
        if run_experiment(e) is None:
            print("失敗したので、ここで止める")
            break

=== C_V22_s2_ctx10_320（モデル v2s2・資料 10 件・320 問）===
[gen] モデル読み込み 1.2 分
[gen] 64/320（1.4 分）
[gen] 128/320（2.7 分）
[gen] 192/320（4.0 分）
[gen] 256/320（5.3 分）
[gen] 320/320（6.6 分）
[gen] done 6.57
→ EM 0.4375（140 問）/ 部分一致 0.5281 / chunk=None 1 / 特定不可 0 / 8.3 分


In [53]:
# # [E7-診断] ① gen_worker.py に __main__ の保護を付ける ② 失敗した worker.log から原因の行だけを取り出す
# import os, re, glob, subprocess
# wp = TOOLS + "/gen_worker.py"
# src = open(wp, encoding="utf-8").read()
# if "__main__" in src:
#     print("① gen_worker.py: すでに __main__ の保護あり")
# else:
#     body = "\n".join(("    " + l) if l.strip() else l for l in src.splitlines())
#     new = "def main():\n" + body + "\n\n\nif __name__ == \"__main__\":\n    main()\n"
#     compile(new, "gen_worker.py", "exec")                       # 構文の確認（壊れていたらここで止まる）
#     open(wp, "w", encoding="utf-8").write(new)
#     print("① gen_worker.py に __main__ の保護を付けた")

# logs = sorted(glob.glob(RUNS + "/C_L10_final_ctx5_320_*/worker.log"), key=os.path.getmtime)
# if not logs:
#     print("② worker.log が見つからない")
# else:
#     lines = open(logs[-1], encoding="utf-8", errors="replace").read().splitlines()
#     print("② %s（%d 行）" % (logs[-1].replace(EVAL_ROOT + "/", ""), len(lines)))
#     pat = re.compile(r"Error|ERROR|error|Traceback|bootstrapping|memory|OOM|CUDA|cuda|401|403|Cannot|not found|No such|Killed|spawn|fork", re.I)
#     shown = 0
#     for i, l in enumerate(lines, 1):
#         if pat.search(l) and "site-packages" not in l and "dist-packages" not in l:
#             print("%4d: %s" % (i, l[:220])); shown += 1
#             if shown >= 40: break
#     if shown == 0:
#         print("（該当行なし。先頭30行を表示）")
#         for i, l in enumerate(lines[:30], 1): print("%4d: %s" % (i, l[:220]))
# print("--- GPU ---")
# print(subprocess.run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip())

① gen_worker.py: すでに __main__ の保護あり
② runs/C_L10_final_ctx5_320_20261115_0643/worker.log（77 行）
   1: INFO 11-15 06:43:43 [api_utils.py:273] non-default args: {'dtype': 'float16', 'seed': 42, 'max_model_len': 8192, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'revision': '9a4d0723a493aa3c2b9570b065744ea16f4a
   8: WARNING 11-15 06:43:44 [vllm.py:1163] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
  11: INFO 11-15 06:43:44 [vllm.py:1392] Cudagraph is disabled under eager mode
  14: WARNING 11-15 06:43:58 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-
  15: (EngineCore pid=10750) INFO 11-15 06:44:17 [core.py:116] Initializing a V1 LLM engine (v0.26.0) with config: model='YAMAHARZ250/qwen3.5-0.8b-rag-lora-final', speculative_config=None, tokenizer='

In [54]:
# [E8 結果の一覧] experiments_results.csv
import pandas as pd
from IPython.display import display
df = pd.read_csv(RESULTS_CSV, encoding="utf-8-sig")
cols = ["name", "status", "gen_n", "gen_EM", "gen_部分一致", "chunk_None", "特定不可", "all_R@5", "all_R@10", "minutes", "gpu"]
df["EM問数"] = (df["gen_EM"] * df["gen_n"]).round().astype("Int64")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(df[cols[:3] + ["EM問数"] + cols[3:]])
print("参考（会社 PC・320問）: L10 127問 / L14 141問 / V21 137問 / V22 142問 / V23 130問。判断の物差し：種の揺れが ±5 問。差が 10 問未満は誤差として扱う")

,name,status,gen_n,EM問数,gen_EM,gen_部分一致,chunk_None,特定不可,all_R@5,all_R@10,minutes,gpu
0,C_L10_final_ctx5_320,error rc=1,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,1.7,"Tesla T4, 15360 MiB"
1,C_L10_final_ctx5_320,error rc=1,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,1.7,"Tesla T4, 15360 MiB"
2,C_L10_final_ctx5_320,ok,320.0,127,0.3969,0.4719,0.0,0.0,0.6909,0.7345,11.9,"Tesla T4, 15360 MiB"
3,C_V22_s2_ctx5_320,ok,320.0,140,0.4375,0.5344,1.0,0.0,0.6909,0.7345,4.4,"Tesla T4, 15360 MiB"
4,C_V21_s1_ctx5_320,ok,320.0,135,0.4219,0.4938,0.0,0.0,0.6909,0.7345,14.9,"Tesla T4, 15360 MiB"
5,C_V23_t0_ctx5_320,ok,320.0,129,0.4031,0.4719,1.0,0.0,0.6909,0.7345,5.2,"Tesla T4, 15360 MiB"
6,C_L10_final_ctx5_320,ok,320.0,127,0.3969,0.4719,0.0,0.0,0.6909,0.7345,5.2,"Tesla T4, 15360 MiB"
7,C_L11_final_ctx7_320,ok,320.0,128,0.4000,0.4719,0.0,0.0,0.6909,0.7345,5.9,"Tesla T4, 15360 MiB"
8,C_L12_final_ctx10_320,ok,320.0,132,0.4125,0.4750,0.0,0.0,0.6909,0.7345,7.9,"Tesla T4, 15360 MiB"
9,C_V22_s2_ctx10_320,ok,320.0,140,0.4375,0.5281,1.0,0.0,0.6909,0.7345,8.3,"Tesla T4, 15360 MiB"


参考（会社 PC・320問）: L10 127問 / L14 141問 / V21 137問 / V22 142問 / V23 130問。判断の物差し：種の揺れが ±5 問。差が 10 問未満は誤差として扱う


In [55]:
# [E9 はい/いいえ問題の答え方] yesno_all_runs.py の移植
def latest_runs():
    # 実験名ごとに、いちばん新しい run フォルダを返す
    d = {}
    for p in sorted(glob.glob(RUNS + "/*/submission.json"), key=os.path.getmtime):
        rd = os.path.dirname(p)
        try:
            nm = json.load(open(rd + "/generation_config.json", encoding="utf-8"))["name"]
        except Exception:
            nm = os.path.basename(rd)
        d[nm] = rd
    return d

yn = {k: v["answers"][0].upper() for k, v in GOLD.items() if is_yn(v["answers"])}
print("はい/いいえ問題: %d 問（YES %d / NO %d）\n" % (len(yn), sum(v == "YES" for v in yn.values()), sum(v == "NO" for v in yn.values())))
print("%-32s %4s %5s %5s %5s %7s %7s %7s" % ("実験", "対象", "はい", "いいえ", "他", "YES正解", "NO正解", "全EM"))
for nm, rd in latest_runs().items():
    sub = json.load(open(rd + "/submission.json", encoding="utf-8"))
    rows = [r for r in sub if r["q_id"] in yn]
    if not rows:
        continue
    y = sum(yesno(r.get("answer")) == "yes" for r in rows)
    n = sum(yesno(r.get("answer")) == "no" for r in rows)
    ok_y = sum(yn[r["q_id"]] == "YES" and yesno(r.get("answer")) == "yes" for r in rows)
    ok_n = sum(yn[r["q_id"]] == "NO" and yesno(r.get("answer")) == "no" for r in rows)
    ty = sum(yn[r["q_id"]] == "YES" for r in rows); tn = len(rows) - ty
    al = [r for r in sub if r["q_id"] in GOLD]
    em = sum(judge(r.get("answer"), GOLD[r["q_id"]]["answers"]) == "完全一致" for r in al) / max(len(al), 1)
    print("%-32s %4d %5d %5d %5d %3d/%-3d %3d/%-3d %7.3f" % (nm[:32], len(rows), y, n, len(rows) - y - n, ok_y, ty, ok_n, tn, em))

はい/いいえ問題: 45 問（YES 22 / NO 23）

実験                                 対象    はい   いいえ     他   YES正解    NO正解     全EM
C_L10_final_ctx5_320               45    40     5     0  21/22    4/23    0.397
C_V22_s2_ctx5_320                  45    22    23     0  17/22   18/23    0.438
C_V21_s1_ctx5_320                  45    28    17     0  18/22   13/23    0.422
C_V23_t0_ctx5_320                  45    33    12     0  18/22    8/23    0.403
C_L11_final_ctx7_320               45    38     7     0  20/22    5/23    0.400
C_L12_final_ctx10_320              45    40     5     0  21/22    4/23    0.412
C_V22_s2_ctx10_320                 45    22    23     0  17/22   18/23    0.438


In [66]:
# [E10 誤り分析] analyze_errors.py と analyze_errors2.py の移植（開発用セットなので、質問・正解・予測を表示してよい）
# ANALYZE = "C_L10_final_ctx5_320"          # ← 分析する実験名
ANALYZE = "C_V22_s2_ctx5_320"

def analyze(name, show=25):
    rd = latest_runs()[name]
    cfgj = json.load(open(rd + "/generation_config.json", encoding="utf-8"))
    k = cfgj["ctx_k"]
    sub = json.load(open(rd + "/submission.json", encoding="utf-8"))
    ctx = {r["q_id"]: r["contexts"][:k] for r in DEV_ROWS}
    rows, cnt, extras = [], collections.Counter(), collections.Counter()
    for r in sub:
        g = GOLD.get(r["q_id"])
        if g is None:
            continue
        pred, golds = r.get("answer") or "", g["answers"]
        gn = [norm(a) for a in golds if norm(a)]
        p = norm(pred)
        j = judge(pred, golds)
        yn_ = is_yn(golds)
        in_ctx = None if yn_ else has_gold(ctx[r["q_id"]], golds)
        sub_t, extra = "", ""
        if j == "完全一致":
            cat = "A_正解"
        elif j == "部分一致":
            cat = "B_形の違い"
            longer = [a for a in gn if a in p and a != p]
            if longer:
                a = max(longer, key=len); i = p.find(a); pre, post = p[:i], p[i + len(a):]
                sub_t = "B1_予測が長い(前に余分)" if pre and not post else "B2_予測が長い(後ろに余分)" if post and not pre else "B3_予測が長い(前後に余分)"
                extra = (pre + "|" if pre else "") + ("|" + post if post else "")
                extras[post or pre] += 1
            else:
                sub_t = "B4_予測が短い(正解の一部)"
        else:
            cat = "X_不正解"
            if pred.startswith("資料からは特定"): sub_t = "X1_特定不可と回答"
            elif yn_: sub_t = "X2_はい/いいえを外した"
            elif r.get("chunk") is None: sub_t = "X3_chunkなし"
            else: sub_t = "X4_その他の誤答"
            if in_ctx is True: cnt["(X のうち 資料に正解あり＝読み違い)"] += 1
            elif in_ctx is False: cnt["(X のうち 資料に正解なし＝検索の取りこぼし)"] += 1
        cnt[cat] += 1
        if sub_t: cnt[sub_t] += 1
        if yn_:
            cnt["(はい/いいえ問題 全体)"] += 1; cnt["(はい/いいえ問題 正解)"] += j == "完全一致"
        rows.append({"q_id": r["q_id"], "source": g.get("source", "?"), "分類": cat, "細分類": sub_t,
                     "正解": " / ".join(golds), "予測": pred, "余分な部分": extra, "予測の長さ": len(p),
                     "正解の長さ(最短)": min((len(a) for a in gn), default=0), "はいいいえ": int(yn_),
                     "資料に正解あり": "" if in_ctx is None else int(in_ctx), "chunk": r.get("chunk")})
    n = len(rows)
    print("== 分類（%s・%d 問・資料%d件）==" % (name, n, k))
    for key in sorted(cnt):
        print("  %-40s %4d  (%.1f%%)" % (key, cnt[key], 100 * cnt[key] / n))
    print("\n== B の余分な部分（多い順・上位20）==")
    for key, v in extras.most_common(20):
        print("  %-20s %d" % (key, v))
    print("\n== 予測の長さ（文字数の中央値）==")
    for c in ("A_正解", "B_形の違い", "X_不正解"):
        ls = sorted(x["予測の長さ"] for x in rows if x["分類"] == c); gs = sorted(x["正解の長さ(最短)"] for x in rows if x["分類"] == c)
        if ls: print("  %-10s 予測 %d / 正解 %d" % (c, ls[len(ls) // 2], gs[len(gs) // 2]))
    print("\n== 出典別 ==")
    by = collections.defaultdict(collections.Counter)
    for r in rows:
        by[str(r["source"])]["n"] += 1; by[str(r["source"])][r["分類"]] += 1
    print("  %-20s %5s %8s %8s %8s" % ("source", "問数", "A正解", "B形", "X不正解"))
    for s, c in sorted(by.items(), key=lambda x: -x[1]["n"]):
        print("  %-20s %5d %7.1f%% %7.1f%% %7.1f%%" % (s, c["n"], 100 * c["A_正解"] / c["n"], 100 * c["B_形の違い"] / c["n"], 100 * c["X_不正解"] / c["n"]))
    print("\n== はい/いいえ問題（正解 × 予測）==")
    m = collections.Counter()
    for r in rows:
        if r["はいいいえ"]:
            g_ = "yes" if "yes" in norm(r["正解"]) else "no"
            m[(g_, yesno(r["予測"]) or "その他")] += 1
    for g_ in ("yes", "no"):
        print("  正解 %-3s → 予測 yes %3d / no %3d / その他 %3d" % (g_, m[(g_, "yes")], m[(g_, "no")], m[(g_, "その他")]))
    def show_(title, cond, kmax):
        sel = [r for r in rows if cond(r)]
        print("\n== %s（%d 件、最大 %d 件表示）==" % (title, len(sel), kmax))
        for r in sel[:kmax]:
            print("  %-10s 正解: %-24s 予測: %s" % (r["q_id"], r["正解"][:24], r["予測"][:40]))
    show_("B4 予測が短い", lambda r: r["細分類"].startswith("B4"), show)
    show_("X2 はい/いいえを外した", lambda r: r["細分類"].startswith("X2"), show)
    show_("X4 その他の誤答（先頭）", lambda r: r["細分類"].startswith("X4"), 30)
    out = rd + "/error_analysis.csv"
    with open(out, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print("\n問題ごとの結果:", out)

analyze(ANALYZE)

== 分類（C_V22_s2_ctx5_320・320 問・資料5件）==
  (X のうち 資料に正解あり＝読み違い)                       61  (19.1%)
  (X のうち 資料に正解なし＝検索の取りこぼし)                   78  (24.4%)
  (はい/いいえ問題 全体)                              45  (14.1%)
  (はい/いいえ問題 正解)                              35  (10.9%)
  A_正解                                      140  (43.8%)
  B1_予測が長い(前に余分)                              3  (0.9%)
  B2_予測が長い(後ろに余分)                             6  (1.9%)
  B3_予測が長い(前後に余分)                             1  (0.3%)
  B4_予測が短い(正解の一部)                            21  (6.6%)
  B_形の違い                                     31  (9.7%)
  X2_はい/いいえを外した                              10  (3.1%)
  X4_その他の誤答                                 139  (43.4%)
  X_不正解                                     149  (46.6%)

== B の余分な部分（多い順・上位20）==
  白赤                   1
  10月11日               1
  フロズン                 1
  製                    1
  chunk_id39788848484439788848484439788848484439788848484439788 1
  1928年                1
  二級建築士木造建築

In [65]:
# [E11 2本の1問ずつの比較] compare_runs.py 相当（基準 → 新しい案）
# BASE_RUN, NEW_RUN = "C_L10_final_ctx5_320", "C_V22_s2_ctx5_320"     # ← 比べる2本
# BASE_RUN, NEW_RUN = "C_V22_s2_ctx5_320", "C_V22_s2_ctx10_320"
BASE_RUN, NEW_RUN = "C_L10_final_ctx5_320", "C_L12_final_ctx10_320"

def compare(a, b):
    runs = latest_runs()
    sa = {r["q_id"]: r for r in json.load(open(runs[a] + "/submission.json", encoding="utf-8"))}
    sb = {r["q_id"]: r for r in json.load(open(runs[b] + "/submission.json", encoding="utf-8"))}
    common = [q for q in sa if q in sb and q in GOLD]
    tally, by_src = collections.Counter(), collections.defaultdict(collections.Counter)
    for q in common:
        ja = judge(sa[q].get("answer"), GOLD[q]["answers"]) == "完全一致"
        jb = judge(sb[q].get("answer"), GOLD[q]["answers"]) == "完全一致"
        key = "両方正解" if ja and jb else "新しい案だけ正解" if jb else "基準だけ正解" if ja else "両方不正解"
        tally[key] += 1
        s = str(GOLD[q].get("source", "?")); by_src[s]["n"] += 1; by_src[s]["基準"] += ja; by_src[s]["新"] += jb
    print("基準 %s → 新しい案 %s（共通 %d 問）" % (a, b, len(common)))
    for key in ("新しい案だけ正解", "基準だけ正解", "両方正解", "両方不正解"):
        print("  %-12s %4d" % (key, tally[key]))
    d = tally["新しい案だけ正解"] - tally["基準だけ正解"]
    print("  差（新 − 基準）: %+d 問 / 答えが分かれた %d 問" % (d, tally["新しい案だけ正解"] + tally["基準だけ正解"]))
    print("\n  %-14s %5s %6s %6s" % ("source", "問数", "基準", "新"))
    for s, c in sorted(by_src.items(), key=lambda x: -x[1]["n"]):
        print("  %-14s %5d %6d %6d" % (s, c["n"], c["基準"], c["新"]))
    print("\n判断の物差し：差が 10 問未満は誤差として扱う（種だけの違いで 5 問動く）")
compare(BASE_RUN, NEW_RUN)

基準 C_L10_final_ctx5_320 → 新しい案 C_L12_final_ctx10_320（共通 320 問）
  新しい案だけ正解       12
  基準だけ正解          7
  両方正解          120
  両方不正解         181
  差（新 − 基準）: +5 問 / 答えが分かれた 19 問

  source            問数     基準      新
  niilc            200     78     82
  jemhopqa         120     49     50

判断の物差し：差が 10 問未満は誤差として扱う（種だけの違いで 5 問動く）


In [67]:
# [E13 読み違いの分析] 資料に正解があるのに外した問題を、「正解の資料が何番目にあるか」「どの資料を引用したか」で分ける
ANALYZE2 = "C_V22_s2_ctx5_320"          # ← 分析する実験名
rd = latest_runs()[ANALYZE2]
k = json.load(open(rd + "/generation_config.json", encoding="utf-8"))["ctx_k"]
sub = {r["q_id"]: r for r in json.load(open(rd + "/submission.json", encoding="utf-8"))}
by_q = {r["q_id"]: r for r in DEV_ROWS}

def gold_chunks(ctxs, golds):
    gn = [norm(a) for a in golds if norm(a)]
    return [(i, c["chunk_id"]) for i, c in enumerate(ctxs, 1) if any(g in norm(c["title"] + c["text"]) for g in gn)]

rank_n, rank_ok = collections.Counter(), collections.Counter()
wrong_cite = collections.Counter()
examples = []
for q, g in GOLD.items():
    if is_yn(g["answers"]) or q not in sub or q not in by_q:
        continue
    ctxs = by_q[q]["contexts"][:k]
    hits = gold_chunks(ctxs, g["answers"])
    if not hits:
        continue                                   # 資料に正解がない問題（検索の取りこぼし）は、ここでは見ない
    first = hits[0][0]
    ok = judge(sub[q].get("answer"), g["answers"]) == "完全一致"
    rank_n[first] += 1
    rank_ok[first] += ok
    if not ok:
        cited_gold = sub[q].get("chunk") in {cid for _, cid in hits}
        wrong_cite["引用した資料に正解がある（取り出しの失敗）" if cited_gold else "引用した資料に正解がない（資料の選び間違い）"] += 1
        if len(examples) < 12:
            examples.append((q, first, g["answers"][0][:20], str(sub[q].get("answer"))[:20], cited_gold))

tot_n, tot_ok = sum(rank_n.values()), sum(rank_ok.values())
print("== %s：資料に正解がある %d 問のうち、正解できたのは %d 問（%.1f%%）==" % (ANALYZE2, tot_n, tot_ok, 100 * tot_ok / max(tot_n, 1)))
print("%-26s %5s %6s %7s" % ("正解の資料が最初に出る位置", "問数", "正解", "正解率"))
for r in range(1, k + 1):
    if rank_n[r]:
        print("%-26s %5d %6d %6.1f%%" % ("%d 番目" % r, rank_n[r], rank_ok[r], 100 * rank_ok[r] / rank_n[r]))
print("\n== 外した %d 問の内訳 ==" % (tot_n - tot_ok))
for key, v in wrong_cite.most_common():
    print("  %-40s %d" % (key, v))
print("\n== 外した問題の例（問題ID / 正解の資料の位置 / 正解 / 予測 / 引用が正解の資料か）==")
for q, first, a, p, cg in examples:
    print("  %-8s %d番目  正解: %-20s 予測: %-20s %s" % (q, first, a, p, "○" if cg else "×"))

== C_V22_s2_ctx5_320：資料に正解がある 190 問のうち、正解できたのは 104 問（54.7%）==
正解の資料が最初に出る位置                 問数     正解     正解率
1 番目                         134     90   67.2%
2 番目                          27     12   44.4%
3 番目                          11      0    0.0%
4 番目                           8      1   12.5%
5 番目                          10      1   10.0%

== 外した 86 問の内訳 ==
  引用した資料に正解がない（資料の選び間違い）                   45
  引用した資料に正解がある（取り出しの失敗）                    41

== 外した問題の例（問題ID / 正解の資料の位置 / 正解 / 予測 / 引用が正解の資料か）==
  dev0001  1番目  正解: スズキ                  予測: すずき                  ×
  dev0014  3番目  正解: 東宝日比谷ビル              予測: 徳丸ビル                 ×
  dev0019  5番目  正解: 1990年                予測: 1991年                ×
  dev0022  1番目  正解: 白                    予測: 赤                    ○
  dev0024  1番目  正解: 青                    予測: 青・白・赤                ○
  dev0025  5番目  正解: 飛騨市                  予測: 関市                   ×
  dev0027  1番目  正解: フクロオオカミ              予測: ティラコレオ               ○
  dev0032 

In [58]:
# [E12 学習データの確認] check_train_data.py の移植（読むだけ・何も書き換えない）
TRAIN_CTX = LORA_ROOT + "/train_contexts_1500.jsonl"      # 検証済みの 1500 行版（画面にあるファイル）
TRAIN_ANS = LORA_ROOT + "/train_answers.json"
YESNO_SET = {"yes", "no", "はい", "いいえ"}
CTX_TOP_K = 5

if not (os.path.exists(TRAIN_CTX) and os.path.exists(TRAIN_ANS)):
    print("学習データが見えない:", TRAIN_CTX)
else:
    tgold = json.load(open(TRAIN_ANS, encoding="utf-8"))
    if isinstance(tgold, list):
        tgold = {x["q_id"]: x for x in tgold}
    dev_q = {norm(r["question"]) for r in DEV_ROWS}          # 開発用セットの質問文（dev_contexts.jsonl から）
    items = load_jsonl(TRAIN_CTX)
    c, qset = collections.Counter(), set()
    for it in items:
        qset.add(norm(it.get("question")))
        ans = tgold.get(it["q_id"], {}).get("answers", [])
        ctx_ = it["contexts"][:CTX_TOP_K]
        if ans and all(norm(a) in YESNO_SET for a in ans):
            c["YES" if norm(ans[0]) in ("yes", "はい") else "NO"] += 1; c["使う"] += 1
            continue
        body = [norm(x["title"] + x["text"]) for x in ctx_]
        if any(norm(a) and norm(a) in b for a in ans for b in body): c["使う"] += 1
        else: c["資料に正解なし"] += 1
    leak = len(qset & dev_q)
    print("[学習データ] %s" % TRAIN_CTX)
    print("  質問 %d / 学習に使う(長さ除外前) %d / 資料に正解なし %d" % (len(items), c["使う"], c["資料に正解なし"]))
    print("  はい・いいえ問題: YES %d / NO %d（全部学習に使われる）" % (c["YES"], c["NO"]))
    print("  資料の件数(1問目): %d / q_id 例: %s" % (len(items[0]["contexts"]), items[0]["q_id"]))
    print("  開発用セットと同じ質問文: %d 件  %s" % (leak, "← 0 でなければ要確認（評価が甘くなる）" if leak else "OK"))
    print("  参考（引き継ぎ書）: 使える 1038 / 資料に正解なし 461 + 長すぎ 1")

[学習データ] /content/drive/MyDrive/LLM講座最終コンペ_lora/train_contexts_1500.jsonl
  質問 1500 / 学習に使う(長さ除外前) 1041 / 資料に正解なし 459
  はい・いいえ問題: YES 147 / NO 143（全部学習に使われる）
  資料の件数(1問目): 5 / q_id 例: tr00001
  開発用セットと同じ質問文: 0 件  OK
  参考（引き継ぎ書）: 使える 1038 / 資料に正解なし 461 + 長すぎ 1


In [59]:
# [確認A] 質問ファイルの中身は読まない・表示しない。件数と q_id の先頭3つだけ表示する
import os, json
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
def ids_only(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, dict):
        ids = [str(v.get("q_id", k)) if isinstance(v, dict) else str(k) for k, v in d.items()]
    else:
        ids = [str(r.get("q_id", r.get("id", ""))) for r in d]
    return len(ids), ids[:3]
for p in (EVAL_ROOT + "/dev_work/questions.json", EVAL_ROOT + "/dev_data/dev_questions_backup.json"):
    if os.path.exists(p):
        n, head = ids_only(p)
        print("%-55s %4d 件 / q_id 先頭3つ: %s" % (p.replace(EVAL_ROOT + "/", ""), n, head))
    else:
        print("%-55s なし" % p.replace(EVAL_ROOT + "/", ""))
print("期待：開発用セットは 320 件・dev0001 形式。q001 形式・301 件は本番の質問")

dev_work/questions.json                                  320 件 / q_id 先頭3つ: ['dev0001', 'dev0002', 'dev0003']
dev_data/dev_questions_backup.json                      なし
期待：開発用セットは 320 件・dev0001 形式。q001 形式・301 件は本番の質問


In [60]:
# [確認B] 本番の質問（q001 形式）で作ってしまったファイルを、使えないように別名に退避する（削除はしない）
import os, json, time
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
def first_id(path):
    d = json.load(open(path, encoding="utf-8"))
    r = (list(d.items())[0] if isinstance(d, dict) else (None, d[0]))
    k, v = r
    return str(v.get("q_id", v.get("id", k))) if isinstance(v, dict) else str(k)
stamp = time.strftime("%Y%m%d_%H%M%S")
for p in (EVAL_ROOT + "/dev_contexts.jsonl", EVAL_ROOT + "/dev_work/questions.json"):
    if not os.path.exists(p):
        print("なし:", p.replace(EVAL_ROOT + "/", "")); continue
    if p.endswith(".jsonl"):
        fid = str(json.loads(open(p, encoding="utf-8").readline()).get("q_id"))
    else:
        fid = first_id(p)
    if fid.startswith("dev"):
        print("開発用セット（%s）なので、そのまま使う: %s" % (fid, p.replace(EVAL_ROOT + "/", "")))
    else:
        new = p + ".NG_not_dev_" + stamp
        os.rename(p, new)
        print("退避した（q_id が %s 形式＝開発用セットではない）→ %s" % (fid, new.replace(EVAL_ROOT + "/", "")))

開発用セット（dev0001）なので、そのまま使う: dev_contexts.jsonl
開発用セット（dev0001）なので、そのまま使う: dev_work/questions.json
